# CottonLens â€” Data Workbench
KalÄ±cÄ± veri defteri. VarsayÄ±lan readiness, dondurulmuÅŸ kaynak kanÄ±tÄ± ve kayÄ±tlÄ± tahminleri inceler; model eÄŸitmez ve API Ã§aÄŸÄ±rmaz. Ham toplama ve yayÄ±n incelemesi aÃ§Ä±k SOURCE_ACTION seÃ§imiyle yapÄ±lÄ±r. Secrets deÄŸerleri Ã§Ä±ktÄ± veya kaynak paketine yazÄ±lmaz.


In [ ]:
# Raw acquisition is separate from historical eligibility. Default is read-only readiness.
SOURCE_ACTION = 'readiness'  # readiness, inventory, acquire, ams_publication_review, observe_ams, live_archive, oncall_status (no training)
YEARS = [2020]  # Expand deliberately after reviewing coverage and API limits.
PROVIDERS = ['ams', 'fas', 'nass']
assert SOURCE_ACTION in ('readiness', 'inventory', 'acquire', 'ams_publication_review', 'observe_ams', 'live_archive', 'oncall_status')
assert all(isinstance(y, int) and 2010 <= y <= 2026 for y in YEARS)
assert set(PROVIDERS) <= {'ams', 'fas', 'nass'}


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT = 'research-free-data-v1'  # Retain this identifier for resumes of EXACTLY this data/source.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path, PurePosixPath
import hashlib, importlib, json, sys, zipfile
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
SOURCE_ZIP = DRIVE_ROOT / 'bundles' / 'source-oncall-exploration-v3-20261003.zip'
EXPECTED_ZIP_SHA256 = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert expected_zip == EXPECTED_ZIP_SHA256, 'Source ZIP sidecar differs from pinned bundle'
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_ZIP_SHA256, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'], 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup, run, INFERENCE
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
from colab_setup import setup_cpu
if SOURCE_ACTION == 'oncall_status':
    import colab_setup
    colab_setup.UV_VERSION = '0.12.14'  # Supports managed Python 3.12.14.
    PYTHON = setup_cpu()
else:
    PYTHON = setup(research=True, data=True)  # Locked managed Python; no global packages.
print('Data-review interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')


In [ ]:
from google.colab import userdata
import os
for name in ('USDA_AMS_API_KEY', 'USDA_FAS_API_KEY', 'USDA_NASS_API_KEY'):
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        os.environ.pop(name, None)
print('Configured credential names:', [n for n in ('USDA_AMS_API_KEY', 'USDA_FAS_API_KEY', 'USDA_NASS_API_KEY') if os.environ.get(n)])


In [ ]:
RAW_USDA = DRIVE_ROOT / 'data' / 'free-usda-raw'
print('Raw archive:', RAW_USDA)
for provider in PROVIDERS:
    print(provider, 'archive directories:', len(list((RAW_USDA / provider).glob('*'))) if (RAW_USDA / provider).exists() else 0)
if SOURCE_ACTION == 'acquire':
    for provider in PROVIDERS:
        assert os.environ.get('USDA_' + provider.upper() + '_API_KEY'), 'Missing Colab Secret for ' + provider
        for year in YEARS:
            if provider == 'ams' and year < 2025:
                print('AMS', year, 'requires archive slug 3004/ESMIS; skipping modern API')
                continue
            command = [PYTHON, '-m', 'cottonlens_ml.sources.usda', '--output', RAW_USDA, '--provider', provider]
            if provider == 'ams':
                command += ['--slug-id', '3804', '--start', f'{year}-08-01' if year == 2025 else f'{year}-01-01', '--end', f'{year}-12-31']
            elif provider == 'fas':
                command += ['--commodity-code', '1404', '--year', str(year)]
            else:
                command += ['--year', str(year)]
            run(command)
if SOURCE_ACTION == 'ams_publication_review':
    table = DRIVE_ROOT / 'reports/ams-diagnostic/spot-quotations-2020-2023.csv'
    manifest = table.with_suffix('.manifest.json')
    assert hashlib.sha256(table.read_bytes()).hexdigest() == '7cc587d313e8c39e5f8bb77f9e4359d9b6275951c54e5d0f26c59e555e832e94', 'Frozen diagnostic table changed'
    destination = DRIVE_ROOT / 'reports/ams-publication-history-v1'
    print('CPU-only evidence review: public publication index + versioned TXT byte comparison; no model fits.')
    print('Browser-compatible archive transport; bounded retries. Verified downloads are reused. No API key is needed.')
    run([PYTHON, '-m', 'cottonlens_ml.sources.ams_publications',
         '--table', table, '--manifest', manifest, '--output', destination])
    result = json.loads((destination / 'publication-content-review.json').read_text(encoding='utf-8'))
    print({key: result[key] for key in ('reference_rows', 'listing_pages', 'matched_dates', 'publication_timezone_verified', 'first_version_reviewed', 'model_eligible')})
    print('Missing metadata dates:', len(result['missing_metadata_dates']), 'Unmatched dates:', len(result['unmatched_dates']))
if SOURCE_ACTION == 'observe_ams':
    destination = DRIVE_ROOT / 'data/observed-public-sources-v1'
    print('Fresh public downloads with UTC completion times; no historical backdating or model admission.')
    for url in ('https://marsapi.ams.usda.gov/services/v3.1/public/listPublishedReports/all',
                'https://www.ams.usda.gov/mnreports/ams_3804.pdf'):
        run([PYTHON, '-m', 'cottonlens_ml.sources.public', 'observe', '--kind', 'ams',
             '--url', url, '--output', destination, '--browser-transport'])
    print('Exact bytes are retained by SHA256; repeated runs create new observations without replacing old versions.')
    print('These captures prove availability by retrieval time only. Separate publication/usage review is still required.')
print('Raw does not mean model eligible. Modern AMS 3804 does not provide pre-August-2025 history.')
print('Historical AMS slug 3004/ESMIS, NASS first releases, FAS originals require publication/vintage review.')
print('No package is automatically compiled or attached to an existing experiment.')


In [ ]:
# Review existing source evidence and saved market predictions; no model fit or API call.
if SOURCE_ACTION == 'readiness':
    EVIDENCE_ROOT = DRIVE_ROOT / 'reports' / 'research-readiness-evidence-v1'
    READINESS = EVIDENCE_ROOT / 'output/reports/free-source-readiness-20261001-v3.json'
    run([PYTHON, '-m', 'cottonlens_ml.research.review',
         '--experiment-root', DRIVE_ROOT / 'experiments/research-market-v1',
         '--evidence-root', EVIDENCE_ROOT, '--readiness', READINESS, '--output', DRIVE_ROOT / 'reports'])
    print('Next: admit a verified source package before any new-information model pilot.')
    print('ML term â€” Baseline: a simple reference prediction used to measure model improvement.')

if SOURCE_ACTION == 'oncall_status':
    # Inspect the frozen small packet only; no API requests, model fits or payload scan.
    import json, hashlib
    packet = DRIVE_ROOT/'data/oncall-exploration-v2-inputs-20261002'
    receipt = json.loads((packet/'input-manifest.json').read_text())
    for name, checksum in receipt['files'].items():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts
        assert hashlib.sha256((packet/name).read_bytes()).hexdigest() == checksum
    run([PYTHON, '-c', "from pathlib import Path; from cottonlens_ml.research.oncall_exploration import read_oncall; import sys; rows,m=read_oncall(Path(sys.argv[1])); print('Pinned On-Call reports:',len(rows),'masked:',m['masked_reports'],'Tier A; release allowed:',m['release_allowed'])", packet/'oncall-2010-2023.csv'])


In [ ]:
if SOURCE_ACTION == 'live_archive':
    # Existing Colab Secrets are inherited by the credential-safe adapters.
    from colab_setup import process_env
    live_env = process_env()
    live_env['PYTHONPATH'] = str(REPO/'ml/src')
    restore = 'from pathlib import Path; import sys; from cottonlens_ml.research.mirror import Mirror; print(Mirror(Path(sys.argv[1]), Path(sys.argv[2])).hydrate())'
    run([PYTHON,'-c',restore,'/content/cottonlens-live-v2',DRIVE_ROOT/'live-baselines-v2'],env=live_env)
    run([PYTHON,'-m','cottonlens_ml.research.live','--store','/content/cottonlens-live-v2','--mirror-root',DRIVE_ROOT], env=live_env)
